# 02 · Preprocess
Face crops (16 frames/video, 224×224) and log-mel spectrograms → saved to Google Drive
under `deepfake_project/processed/<dataset>/`. Raw data from notebook 01 must be present
in this session. Re-running skips files that are already done.

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = ""  # e.g. "https://github.com/<your-username>/robust-multimodal-deepfake.git"

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

## Smoke test: 10 DFDC videos

In [ ]:
!python -m deepfake.preprocessing.run dfdc --limit 10 --workers 2

## Full runs

In [ ]:
# DFDC sample: all ~400 videos
!python -m deepfake.preprocessing.run dfdc --workers 2

In [ ]:
# ASVspoof: balanced subset (the eval split alone has ~71k files)
!python -m deepfake.preprocessing.run asvspoof --per-class 2500 --workers 2

In [ ]:
# FakeAVCeleb: balanced across the 4 categories (only 500 real videos exist)
import os
if os.path.exists(f"{cfg['paths']['manifests']}/fakeavceleb.csv"):
    !python -m deepfake.preprocessing.run fakeavceleb --per-class 400 --workers 2

## Inspect the results

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from deepfake.preprocessing.faces import load_faces

DATASET = "dfdc"
root = Path(cfg["paths"]["processed"]) / DATASET
idx = pd.read_csv(root / "index.csv", keep_default_na=False)
print(idx.groupby(["split", "label"]).size().unstack(fill_value=0))
print("errors:", (idx.error != "").sum())
v = idx[idx.n_frames > 0]
print(f"face detection rate: {v.n_detected.sum() / v.n_frames.sum():.1%}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for row, label in enumerate([0, 1]):
    sample = idx[(idx.label == label) & (idx.faces_path != "")].sample(1, random_state=0).iloc[0]
    faces, detected = load_faces(root / sample.faces_path)
    for j in range(5):
        axes[row, j].imshow(faces[j * 3]); axes[row, j].axis("off")
    axes[row, 0].set_title(("REAL" if label == 0 else "FAKE") + f" · {sample.sample_id}", loc="left")
    if sample.mel_path:
        axes[row, 5].imshow(np.load(root / sample.mel_path).astype(float), origin="lower", aspect="auto")
        axes[row, 5].set_title("log-mel")
    axes[row, 5].axis("off")
plt.tight_layout()

## Dataset sanity check (what the models will receive)

In [ ]:
from deepfake.datasets import MultimodalDataset
ds = MultimodalDataset(root / "index.csv", "train", cfg, train=True)
item = ds[0]
print(len(ds), "train samples")
print({k: tuple(v.shape) for k, v in item.items()})